# 02 - XGBoost / LightGBM Modeling

Ce notebook utilise **exactement la meme logique que `train.py`** pour garantir la coherence des resultats.

## Problemes corriges par rapport a la version precedente:
1. **Colonnes non-numeriques** - `_coerce_object_columns()` convertit toutes les colonnes en numerique
2. **Data leakage** - Lags/windows < HORIZON sont supprimes
3. **Split temporel** - Utilise `split_time_series()` identique a train.py
4. **Metriques** - Formules identiques a train.py


In [ ]:
import sys
import re
import warnings
from pathlib import Path
from math import sqrt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')

# Setup - SAME AS train.py
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Import project modules - SAME AS train.py
from src.data.features import build_feature_pipeline
from src.config import (
    FEATURE_EXCLUDE, PREDICT_LAGS, PREDICT_WINDOWS,
    GROUP_COLS, CATEGORICAL_FEATURES, TRAIN_CSV,
)

try:
    from src.data.clean import clean_dataframe
    CLEAN_AVAILABLE = True
except ImportError:
    CLEAN_AVAILABLE = False

print(f"Imports OK")
print(f"PREDICT_LAGS: {PREDICT_LAGS}")
print(f"PREDICT_WINDOWS: {PREDICT_WINDOWS}")

## 1. Configuration (identique a train.py)

In [ ]:
# Configuration - SAME AS train.py defaults
HORIZON = 14  # Same default as train.py
N_CV_SPLITS = 5

# Filter lags/windows >= horizon - SAME AS train.py
LAGS = [l for l in PREDICT_LAGS if l >= HORIZON] or [max(PREDICT_LAGS)]
WINDOWS = [w for w in PREDICT_WINDOWS if w >= HORIZON] or [max(PREDICT_WINDOWS)]

print(f"HORIZON: {HORIZON}")
print(f"LAGS (>= horizon): {LAGS}")
print(f"WINDOWS (>= horizon): {WINDOWS}")

## 2. Chargement des Donnees (identique a train.py load_and_prepare)

In [ ]:
# Find data - SAME LOGIC AS train.py
features_path = TRAIN_CSV if TRAIN_CSV.exists() else None
if features_path is None:
    fallback = PROJECT_ROOT / "data" / "processed" / "uploaded_generated_training_10950_features.csv"
    if fallback.exists():
        features_path = fallback
    else:
        sample = PROJECT_ROOT / "data" / "processed" / "train_features.csv"
        if sample.exists():
            features_path = sample

if features_path is None:
    raise FileNotFoundError("No data file found")

print(f"Using: {features_path}")

In [ ]:
# Load and prepare - SAME AS train.py load_and_prepare()
df = pd.read_csv(features_path, parse_dates=['date'])
print(f"Shape initiale: {df.shape}")

# Apply cleaning - SAME AS train.py
if CLEAN_AVAILABLE:
    df = clean_dataframe(df, date_col='date', value_col='value', 
                         fill_strategy='ffill', outlier_threshold=None)
    print("clean_dataframe() applique")

# Remove short-horizon columns - SAME AS train.py
short_col_re = re.compile(
    r'^(?:lag|roll_mean|roll_std|roll_min|roll_max|roll_range|roll_cv|ewma|lag_diff|lag_ratio)_(\d+)'
)
cols_to_drop = [c for c in df.columns if (m := short_col_re.match(c)) and int(m.group(1)) < HORIZON]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped {len(cols_to_drop)} short-horizon columns")

# Remap columns - SAME AS train.py
if 'store_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id': 'store_id'})

# Handle on_promo - SAME AS train.py
if 'on_promo' in df.columns:
    df['on_promo'] = df['on_promo'].map(
        {True: 1, False: 0, 'True': 1, 'False': 0, 1: 1, 0: 0}
    ).fillna(0).astype(int)

# Detect groups - SAME AS train.py
available_groups = [c for c in GROUP_COLS if c in df.columns]
print(f"Group columns: {available_groups}")

# Sort - SAME AS train.py
sort_cols = available_groups + ['date']
df = df.sort_values(sort_cols).reset_index(drop=True)

# Detect categoricals - SAME AS train.py
cat_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns]
print(f"Categorical columns: {cat_cols}")

## 3. Feature Engineering (identique a train.py)

In [ ]:
# Build features - SAME AS train.py
df, encoders = build_feature_pipeline(
    df,
    lags=LAGS,
    windows=WINDOWS,
    group_cols=available_groups if available_groups else None,
    categorical_cols=cat_cols if cat_cols else None,
    is_train=True,
    horizon=HORIZON,
)

# Drop NaN - SAME AS train.py
before = len(df)
df = df.dropna().reset_index(drop=True)
after = len(df)
print(f"Dropped {before - after} NaN rows")
print(f"Shape finale: {df.shape}")
print(f"Target stats: mean={df['value'].mean():.2f}, std={df['value'].std():.2f}")

## 4. Split Temporel (identique a train.py split_time_series)

In [ ]:
# Split - EXACT COPY of train.py split_time_series()
def split_time_series(df, horizon, date_col='date', target_col='value', exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = set()
    exclude_cols = set(exclude_cols) | {target_col, date_col}
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    
    unique_dates = sorted(df[date_col].unique())
    if horizon >= len(unique_dates):
        horizon = max(1, len(unique_dates) // 5)
        print(f"Adjusted horizon to {horizon}")
    
    cutoff_date = unique_dates[-horizon]
    train_mask = df[date_col] < cutoff_date
    test_mask = df[date_col] >= cutoff_date
    
    X_train = df.loc[train_mask, feature_cols].copy()
    y_train = df.loc[train_mask, target_col].copy()
    X_test = df.loc[test_mask, feature_cols].copy()
    y_test = df.loc[test_mask, target_col].copy()
    dates_test = df.loc[test_mask, date_col].copy()
    
    return X_train, y_train, X_test, y_test, dates_test

X_train, y_train, X_test, y_test, dates_test = split_time_series(
    df, horizon=HORIZON, exclude_cols=FEATURE_EXCLUDE
)

print(f"Train: {len(X_train)} samples")
print(f"Test: {len(X_test)} samples")
print(f"Features: {X_train.shape[1]}")

In [ ]:
# Create validation set - SAME AS train.py
val_size = max(int(len(X_train) * 0.15), HORIZON)
X_tr = X_train.iloc[:-val_size]
y_tr = y_train.iloc[:-val_size]
X_val = X_train.iloc[-val_size:]
y_val = y_train.iloc[-val_size:]

print(f"Train/Val split: {len(X_tr)} / {len(X_val)}")

## 5. **CRITICAL**: Coercion des colonnes (comme train.py)

In [ ]:
# CRITICAL: This is the key fix - EXACT COPY from train.py
def _coerce_object_columns(df_in):
    """Convert object/categorical columns to numeric - SAME AS train.py"""
    df_out = df_in.copy()
    for c in df_out.columns:
        if pd.api.types.is_object_dtype(df_out[c].dtype) or isinstance(df_out[c].dtype, pd.CategoricalDtype):
            try:
                df_out[c] = pd.Categorical(df_out[c]).codes
            except Exception:
                df_out[c] = pd.to_numeric(df_out[c], errors='coerce').fillna(0).astype(float)
    return df_out

# Apply coercion - SAME AS train.py
X_tr = _coerce_object_columns(X_tr)
X_val = _coerce_object_columns(X_val)
X_test = _coerce_object_columns(X_test)
X_train = _coerce_object_columns(X_train)

# Also prepare X_all for CV
exclude_cols = set(FEATURE_EXCLUDE) | {'value', 'date'}
feature_cols = [c for c in df.columns if c not in exclude_cols]
X_all = _coerce_object_columns(df[feature_cols].copy())
y_all = df['value'].copy()

print(f"Coercion complete")
print(f"X_train dtypes: {X_train.dtypes.value_counts().to_dict()}")

## 6. Entrainement (identique a train.py)

In [ ]:
# Default params - SAME AS train.py _default_lgbm_params()
LGBM_PARAMS = {
    'n_estimators': 1000,
    'max_depth': 8,
    'learning_rate': 0.03,
    'num_leaves': 24,
    'min_child_samples': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbosity': -1,
    'n_jobs': -1,
}

XGB_PARAMS = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.05,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'early_stopping_rounds': 50,
    'verbosity': 0,
}

In [ ]:
# Train LightGBM - SAME AS train.py train_lgbm()
try:
    from lightgbm import LGBMRegressor, early_stopping, log_evaluation
    
    model_lgbm = LGBMRegressor(**LGBM_PARAMS)
    model_lgbm.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(50), log_evaluation(-1)],
    )
    print(f"LightGBM best iteration: {model_lgbm.best_iteration_}")
    USE_LGBM = True
except ImportError:
    print("LightGBM not available")
    model_lgbm = None
    USE_LGBM = False

In [ ]:
# Train XGBoost - SAME AS train.py train_xgb()
from xgboost import XGBRegressor

model_xgb = XGBRegressor(**XGB_PARAMS)
model_xgb.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"XGBoost best iteration: {model_xgb.best_iteration}")

## 7. Evaluation (identique a train.py evaluate_model)

In [ ]:
# Metrics - SAME AS train.py evaluate_model()
def evaluate_model(model, X_train, y_train, X_test, y_test, X_all, y_all, n_cv_splits=5):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Train metrics
    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmse_train = sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)
    bias_train = float(np.mean(y_pred_train - y_train))
    
    # Test metrics
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = sqrt(mean_squared_error(y_test, y_pred_test))
    r2_test = r2_score(y_test, y_pred_test)
    bias_test = float(np.mean(y_pred_test - y_test))
    
    # Bias-corrected
    y_pred_test_corrected = y_pred_test - bias_test
    mae_test_corrected = mean_absolute_error(y_test, y_pred_test_corrected)
    rmse_test_corrected = sqrt(mean_squared_error(y_test, y_pred_test_corrected))
    r2_test_corrected = r2_score(y_test, y_pred_test_corrected)
    
    # Safe MAPE - SAME AS train.py
    y_test_arr = np.array(y_test)
    mask = np.abs(y_test_arr) > 1.0
    if mask.sum() > 0:
        mape_test = float(np.mean(np.abs((y_test_arr[mask] - y_pred_test[mask]) / y_test_arr[mask])) * 100)
    else:
        mape_test = None
    
    # sMAPE - SAME AS train.py
    denom = (np.abs(y_test_arr) + np.abs(y_pred_test)) / 2
    denom_safe = np.where(denom > 0, denom, 1.0)
    smape_test = float(np.mean(np.abs(y_test_arr - y_pred_test) / denom_safe) * 100)
    
    # CV - SAME AS train.py
    tscv = TimeSeriesSplit(n_splits=min(n_cv_splits, max(2, len(X_all) // 100)))
    cv_scores = {'mae': [], 'r2': []}
    
    for train_idx, val_idx in tscv.split(X_all):
        X_tr_cv, X_val_cv = X_all.iloc[train_idx], X_all.iloc[val_idx]
        y_tr_cv, y_val_cv = y_all.iloc[train_idx], y_all.iloc[val_idx]
        
        try:
            from lightgbm import LGBMRegressor
            model_cv = LGBMRegressor(
                n_estimators=500, max_depth=6, learning_rate=0.05,
                random_state=42, verbosity=-1, n_jobs=-1,
            )
            model_cv.fit(
                X_tr_cv, y_tr_cv,
                eval_set=[(X_val_cv, y_val_cv)],
                callbacks=[early_stopping(30), log_evaluation(-1)],
            )
        except:
            model_cv = XGBRegressor(
                n_estimators=500, max_depth=6, learning_rate=0.05,
                random_state=42, early_stopping_rounds=30, verbosity=0,
            )
            model_cv.fit(X_tr_cv, y_tr_cv, eval_set=[(X_val_cv, y_val_cv)], verbose=False)
        
        y_val_pred = model_cv.predict(X_val_cv)
        cv_scores['mae'].append(float(mean_absolute_error(y_val_cv, y_val_pred)))
        cv_scores['r2'].append(float(r2_score(y_val_cv, y_val_pred)))
    
    overfit_ratio = rmse_test / rmse_train if rmse_train > 0 else float('inf')
    
    return {
        'MAE_train': mae_train, 'RMSE_train': rmse_train, 'R2_train': r2_train, 'Bias_train': bias_train,
        'MAE_test': mae_test, 'RMSE_test': rmse_test, 'R2_test': r2_test, 'Bias_test': bias_test,
        'MAE_test_corrected': mae_test_corrected, 'RMSE_test_corrected': rmse_test_corrected, 'R2_test_corrected': r2_test_corrected,
        'MAPE_test': mape_test, 'sMAPE_test': smape_test,
        'overfit_ratio': overfit_ratio,
        'CV_mae_mean': float(np.mean(cv_scores['mae'])), 'CV_mae_std': float(np.std(cv_scores['mae'])),
        'CV_r2_mean': float(np.mean(cv_scores['r2'])), 'CV_r2_std': float(np.std(cv_scores['r2'])),
    }

In [ ]:
# Evaluate LightGBM
if USE_LGBM:
    metrics_lgbm = evaluate_model(model_lgbm, X_train, y_train, X_test, y_test, X_all, y_all)
    
    print("\n" + "="*60)
    print("LightGBM")
    print("="*60)
    print(f"  Train  -> MAE={metrics_lgbm['MAE_train']:.4f}  RMSE={metrics_lgbm['RMSE_train']:.4f}  R2={metrics_lgbm['R2_train']:.4f}")
    print(f"  Test   -> MAE={metrics_lgbm['MAE_test']:.4f}  RMSE={metrics_lgbm['RMSE_test']:.4f}  R2={metrics_lgbm['R2_test']:.4f}")
    print(f"  Test*  -> MAE={metrics_lgbm['MAE_test_corrected']:.4f}  RMSE={metrics_lgbm['RMSE_test_corrected']:.4f}  R2={metrics_lgbm['R2_test_corrected']:.4f}  (* bias-corrected)")
    print(f"  Bias   -> train={metrics_lgbm['Bias_train']:.4f}  test={metrics_lgbm['Bias_test']:.4f}")
    if metrics_lgbm['MAPE_test']:
        print(f"  MAPE   -> {metrics_lgbm['MAPE_test']:.2f}%")
    print(f"  sMAPE  -> {metrics_lgbm['sMAPE_test']:.2f}%")
    print(f"  Overfit ratio: {metrics_lgbm['overfit_ratio']:.2f}")
    print(f"  CV     -> MAE={metrics_lgbm['CV_mae_mean']:.4f} (+/- {metrics_lgbm['CV_mae_std']:.4f})")
    print(f"  CV     -> R2={metrics_lgbm['CV_r2_mean']:.4f} (+/- {metrics_lgbm['CV_r2_std']:.4f})")
    print("="*60)

In [ ]:
# Evaluate XGBoost
metrics_xgb = evaluate_model(model_xgb, X_train, y_train, X_test, y_test, X_all, y_all)

print("\n" + "="*60)
print("XGBoost")
print("="*60)
print(f"  Train  -> MAE={metrics_xgb['MAE_train']:.4f}  RMSE={metrics_xgb['RMSE_train']:.4f}  R2={metrics_xgb['R2_train']:.4f}")
print(f"  Test   -> MAE={metrics_xgb['MAE_test']:.4f}  RMSE={metrics_xgb['RMSE_test']:.4f}  R2={metrics_xgb['R2_test']:.4f}")
print(f"  Test*  -> MAE={metrics_xgb['MAE_test_corrected']:.4f}  RMSE={metrics_xgb['RMSE_test_corrected']:.4f}  R2={metrics_xgb['R2_test_corrected']:.4f}  (* bias-corrected)")
print(f"  Bias   -> train={metrics_xgb['Bias_train']:.4f}  test={metrics_xgb['Bias_test']:.4f}")
if metrics_xgb['MAPE_test']:
    print(f"  MAPE   -> {metrics_xgb['MAPE_test']:.2f}%")
print(f"  sMAPE  -> {metrics_xgb['sMAPE_test']:.2f}%")
print(f"  Overfit ratio: {metrics_xgb['overfit_ratio']:.2f}")
print(f"  CV     -> MAE={metrics_xgb['CV_mae_mean']:.4f} (+/- {metrics_xgb['CV_mae_std']:.4f})")
print(f"  CV     -> R2={metrics_xgb['CV_r2_mean']:.4f} (+/- {metrics_xgb['CV_r2_std']:.4f})")
print("="*60)

## 8. Feature Importance

In [ ]:
# Feature importance - SAME AS train.py get_feature_importance()
def get_feature_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        imp = model.feature_importances_
    else:
        return pd.DataFrame()
    
    fi = pd.DataFrame({'feature': feature_names, 'importance': imp})
    fi = fi.sort_values('importance', ascending=False).reset_index(drop=True)
    return fi

model_to_use = model_lgbm if USE_LGBM else model_xgb
fi = get_feature_importance(model_to_use, list(X_train.columns))

print("\nTop 15 Features:")
print(fi.head(15).to_string(index=False))

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))
top_fi = fi.head(20)
ax.barh(top_fi['feature'], top_fi['importance'])
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Visualisation

In [ ]:
# Predictions vs Actual
y_pred = model_to_use.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax1 = axes[0]
ax1.scatter(y_test, y_pred, alpha=0.5)
max_val = max(y_test.max(), y_pred.max())
ax1.plot([0, max_val], [0, max_val], 'r--', label='Perfect')
ax1.set_xlabel('Actual')
ax1.set_ylabel('Predicted')
r2 = metrics_lgbm['R2_test'] if USE_LGBM else metrics_xgb['R2_test']
ax1.set_title(f'Predictions vs Actual (R2={r2:.4f})')
ax1.legend()

# Residuals
ax2 = axes[1]
residuals = y_pred - y_test.values
ax2.hist(residuals, bins=50, edgecolor='black')
ax2.axvline(x=0, color='red', linestyle='--')
ax2.set_xlabel('Residual')
ax2.set_ylabel('Count')
ax2.set_title(f'Residuals Distribution (Bias={np.mean(residuals):.2f})')

plt.tight_layout()
plt.show()

## 10. Resume Final

In [ ]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Model        : {'LightGBM' if USE_LGBM else 'XGBoost'}")
print(f"Horizon      : {HORIZON} days")
print(f"Features     : {X_train.shape[1]}")
print(f"Train samples: {len(X_train)}")
print(f"Test samples : {len(X_test)}")

m = metrics_lgbm if USE_LGBM else metrics_xgb
print(f"\nTest Metrics:")
print(f"  MAE  : {m['MAE_test']:.4f}")
print(f"  RMSE : {m['RMSE_test']:.4f}")
print(f"  R2   : {m['R2_test']:.4f}")
if m['MAPE_test']:
    print(f"  MAPE : {m['MAPE_test']:.2f}%")
print(f"  sMAPE: {m['sMAPE_test']:.2f}%")

print(f"\nCV Metrics ({N_CV_SPLITS} folds):")
print(f"  MAE : {m['CV_mae_mean']:.4f} (+/- {m['CV_mae_std']:.4f})")
print(f"  R2  : {m['CV_r2_mean']:.4f} (+/- {m['CV_r2_std']:.4f})")

print(f"\nOverfit ratio: {m['overfit_ratio']:.2f}")

# Verdict - SAME AS train.py
r2_test = m['R2_test']
if r2_test >= 0.85:
    verdict = "Excellent model -- strong predictive power."
elif r2_test >= 0.7:
    verdict = "Good model -- explains significant variance."
elif r2_test >= 0.5:
    verdict = "Moderate model -- captures key patterns."
elif r2_test >= 0.3:
    verdict = "Weak model -- limited signal captured."
else:
    verdict = "Poor model -- needs better data or features."

print(f"\nVERDICT: {verdict}")
print("="*60)